In [385]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
oecd = pd.read_csv('/Users/riley/Desktop/Data Case Competition/OECD Dataset.xlsx - complete_p4d3_df.csv')

primary = ['year', 'usd_disbursements_defl', 'project_description', 'expected_duration']
geography = ['country', 'region', 'region_macro']
funder = ['organization_name', 'Donor_country']
aiding = ['Sector', 'subsector']

oecd = oecd[primary + geography + funder + aiding]
oecd = oecd.dropna(subset=['usd_disbursements_defl']) # drop null disbursement

/var/folders/gh/dn12f24577v7qm098b5qxv580000gn/T/ipykernel_61492/2726326578.py:4: DtypeWarning: Columns (0,5,7,10,11,12,13,17,21,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  oecd = pd.read_csv('/Users/riley/Desktop/Data Case Competition/OECD Dataset.xlsx - complete_p4d3_df.csv')


In [386]:
# foundations like Sabanci, Cartier, and Fondation De Luxembourg donate to multiple sectors at once
# for ease of use, I duplicated each row for each sector, then divided the disbursement evenly

sector_col = 'Sector'
money_col = 'usd_disbursements_defl'

mask = oecd[sector_col].astype(str).str.contains(';', na=False)

multi_df = oecd[mask].copy()
oecd = oecd[~mask].copy()

multi_df[sector_col] = multi_df[sector_col].astype(str).str.split(';')
multi_df[sector_col] = multi_df[sector_col].apply(
    lambda codes: [code.strip() for code in codes]
)

multi_df['num_sectors_split'] = multi_df[sector_col].apply(len)
multi_df[money_col] = multi_df[money_col] / multi_df['num_sectors_split']
multi_df = multi_df.explode(sector_col)

oecd['Sector'] = pd.to_numeric(oecd['Sector'], errors='coerce').astype('Int64')

oecd['Sector'].unique()


<IntegerArray>
[ 720,  410,  930,  150, <NA>,  160,  120,  998,  430,  910,  110,  220,  230,
  740,  310,  130,  240,  140,  320,  510,  210,  730,  250,  330,  520,  600,
  530,  122,  151,  121,  311,  123,  112,  114,  111,  113,  313,  232,  231,
  152,  312,  332,  321,  322,  331,  236,  323]
Length: 47, dtype: Int64

In [387]:
valid_codes = oecd['Sector'].unique().dropna()
counts = oecd.groupby('Sector').size().reset_index(name='num_projects')

print(f'length valid codes {len(valid_codes)}')
print(counts)

length valid codes 46
    Sector  num_projects
0      110         12306
1      111           992
2      112          1485
3      113           755
4      114          1074
5      120         14565
6      121          2619
7      122         11528
8      123           883
9      130          7044
10     140          3309
11     150          4713
12     151         14542
13     152           211
14     160          8491
15     210           433
16     220           293
17     230           383
18     231           228
19     232           744
20     236             8
21     240          1623
22     250           946
23     310          1554
24     311          3970
25     312            99
26     313           520
27     320           954
28     321            78
29     322            49
30     323             7
31     330            69
32     331            15
33     332            26
34     410          6006
35     430          3069
36     510            38
37     520          1530
38 

In [388]:
# groupings the sectors into broader sector purposes

code_to_purpose = {
    110:'education', 111:'education', 112:'education', 113:'education', 114:'education',

    120:'health', 121:'health', 122:'health', 123:'health', 130:'health',

    140:'water_and_sanitation',

    150:'governance_and_social', 151:'governance_and_social', 152:'governance_and_social', 160:'governance_and_social',

    210:'infrastructure', 220:'infrastructure',

    230:'energy', 231:'energy', 232:'energy', 236:'energy',

    240:'economic_development', 250:'economic_development',
    320:'economic_development', 321:'economic_development', 322:'economic_development', 323:'economic_development',
    330:'economic_development', 331:'economic_development', 332:'economic_development',

    310:'environment_and_agriculture', 311:'environment_and_agriculture',
    312:'environment_and_agriculture', 313:'environment_and_agriculture',
    410:'environment_and_agriculture',

    720:'humanitarian_and_disaster', 730:'humanitarian_and_disaster', 740:'humanitarian_and_disaster',

    510:'aid_and_debt', 520:'aid_and_debt', 530:'aid_and_debt', 600:'aid_and_debt',

    430:'other', 910:'other', 930:'other', 998:'other'
}

oecd['broad_sector'] = oecd['Sector'].map(code_to_purpose)

In [389]:
def sector_summary(table):
    return table.groupby('broad_sector', dropna=False)['usd_disbursements_defl'].agg(broad_counts='size',total_disbursements='sum').reset_index()
sector_summary(oecd)

,broad_sector,broad_counts,total_disbursements
0,aid_and_debt,1601,273.078609
1,economic_development,3767,5747.352309
2,education,16612,7656.494827
3,energy,1363,1423.821988
4,environment_and_agriculture,12149,7935.593099
5,governance_and_social,27957,8320.991465
6,health,36639,27101.589033
7,humanitarian_and_disaster,2320,2027.634310
8,infrastructure,726,457.215655
9,other,4885,2930.498439


In [390]:
oecd['broad_sector'].isnull().sum() + (oecd['broad_sector'] == '').sum() # number of philanthropy projects without a tagged sector

np.int64(4107)

In [391]:
# machine learning to find the most important words for each broader category
# used to filter the project descriptions of untagged words and categorize them

text_col = 'project_description'
label_col = 'broad_sector'

df_labeled = oecd.dropna(subset=[text_col, label_col]).copy()

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    min_df=5,
    max_df=0.6,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df_labeled[text_col].astype(str))
words = np.array(vectorizer.get_feature_names_out())

indicator_words = {}

for category in df_labeled[label_col].unique():
    category_mask = df_labeled[label_col] == category
    
    category_scores = X[category_mask].mean(axis=0).A1
    other_scores = X[~category_mask].mean(axis=0).A1
    
    exclusivity_score = category_scores / (other_scores + 0.001)
    
    top_indices = exclusivity_score.argsort()[::-1][:15]
    
    indicator_words[category] = words[top_indices].tolist()

indicator_words

{'humanitarian_and_disaster': ['disaster',
  'relief',
  'humanitarian',
  'earthquake',
  'emergency',
  'disaster relief',
  'affected',
  'provide humanitarian',
  'relief efforts',
  'roja',
  'cruz roja',
  'humanitarian assistance',
  'flood',
  'cruz',
  'roja mexicana'],
 'environment_and_agriculture': ['conservation',
  'programme conservation',
  'conservation science',
  'science description',
  'environment',
  'agricultural',
  'farmers',
  'smallholder',
  'programme environment',
  'climate',
  'climate environment',
  'apes',
  'agriculture',
  'science',
  'gibbons'],
 'other': ['grants',
  'active cititzenship',
  'cititzenship',
  'active',
  'grant',
  'prize heard',
  'heard competition',
  'village',
  'heard',
  'urban95',
  'self employment',
  'employment program',
  'competition',
  'town',
  'prize'],
 'governance_and_social': ['human rights',
  'rights',
  'civic',
  'justice',
  'provide general',
  'human',
  'general support',
  'civic engagement',
  'vio

In [392]:
# additional words picked from a list of project descriptions that were not initially categorized by the function
# rinse and repeat process

indicator_words['humanitarian_and_disaster'].extend([
    'crisis', 'recovery', 'shelter', 'resilience', 'volcanic',
    'ciclone', 'cyclone', 'urgência', 'urgence', 'alimentaire',
    'quarantine', 'postdisaster', 'affected', 'relocation', 'response'
])

indicator_words['environment_and_agriculture'].extend([
    'plastic', 'pollution', 'recycling', 'waste', 'emissions',
    'oceans', 'fisheries', 'fishery', 'deforestation', 'agroecological',
    'seeds', 'farming', 'trees', 'mangue', 'ecosystems'
])

indicator_words['other'].extend([
    'charitable', 'donation', 'membership', 'various', 'unrestricted',
    'core', 'operating', 'support', 'fellowship', 'research',
    'coordination', 'philanthropy', 'foundation', 'prize', 'general'
])

indicator_words['governance_and_social'].extend([
    'women', 'girls', 'gender', 'trans', 'lgbtqia',
    'rights', 'advocacy', 'accountability', 'democracy', 'transparency',
    'justice', 'law', 'protection', 'community', 'participation'
])

indicator_words['health'].extend([
    'palliative', 'medical', 'psychological', 'covid', 'corona',
    'patients', 'tuberculosis', 'nutrition', 'maternal', 'vaccination',
    'medicines', 'hospital', 'clinic', 'healthcare', 'malades'
])

indicator_words['education'].extend([
    'learning', 'training', 'scholarships', 'bursaries', 'learners',
    'teachers', 'curriculum', 'classroom', 'livres', 'scolaire',
    'upskilling', 'students', 'stem', 'grammar', 'mathematics'
])

indicator_words['infrastructure'].extend([
    'transport', 'bicycles', 'construction', 'rehabilitation', 'renovation',
    'repair', 'housing', 'building', 'equipment', 'facilities',
    'technology', 'digital', 'internet', 'clinic', 'polyclinique'
])

indicator_words['energy'].extend([
    'electricity', 'electrification', 'grid', 'solar', 'power',
    'renewable', 'energy', 'decarbonizing', 'decarbonization', 'emissions',
    'appliances', 'transition', 'clean', 'shipping', 'fuel'
])

indicator_words['economic_development'].extend([
    'mobility', 'career', 'income', 'entrepreneurial', 'entrepreneurship',
    'microenterprise', 'market', 'artisan', 'handicraft', 'employment',
    'jobs', 'fintech', 'startups', 'businesses', 'loans'
])

indicator_words['water_and_sanitation'].extend([
    'hygiene', 'toilets', 'latrines', 'potable', 'wastewater',
    'drainage', 'robinets', 'toilettes', 'water', 'sanitation',
    'wash', 'eau', 'safe', 'clean', 'supply'
])

indicator_words['aid_and_debt'].extend([
    'food', 'families', 'despensa', 'alimentation', 'alimentaire',
    'alimentarios', 'kits', 'grain', 'cereal', 'poverty',
    'pobreza', 'marginacion', 'donativo', 'commodity', 'assistance'
])

In [393]:
def assign_broad_sector(row):
    if pd.notna(row['broad_sector']):
        return row['broad_sector']
    
    description = str(row['project_description']).lower()
    
    scores = {}
    
    for category, words in indicator_words.items():
        score = 0
        
        for word in words:
            if word.lower() in description:
                score += 1
        
        scores[category] = score
    
    best_category = max(scores, key=scores.get)
    
    if scores[best_category] == 0:
        return np.nan
    
    return best_category

oecd_updated = oecd.copy()
oecd_updated['broad_sector'] = oecd_updated.apply(assign_broad_sector, axis=1)

In [394]:
print('before null categorization:', oecd['broad_sector'].isnull().sum() + (oecd['broad_sector'] == '').sum()) # before null categorization
print('after null categorization:', oecd_updated['broad_sector'].isnull().sum() + (oecd_updated['broad_sector'] == '').sum()) # after null categorization
print(sector_summary(oecd_updated))
print('uncategorized projects with null descriptions:', len(oecd_updated[oecd_updated['broad_sector'].isnull() & oecd_updated['project_description'].isnull()]))
# this is the amount of money unaccounted
# the project description is null, or insufficient in categorizing the broader category

before null categorization: 4107
after null categorization: 1731
                   broad_sector  broad_counts  total_disbursements
0                  aid_and_debt          1642           296.999803
1          economic_development          3817          5763.412615
2                     education         16845          7769.019079
3                        energy          1403          1434.186233
4   environment_and_agriculture         12246          7977.395595
5         governance_and_social         28104          8337.317484
6                        health         36812         27538.565011
7     humanitarian_and_disaster          2453          2159.363519
8                infrastructure           963           606.341395
9                         other          6053          3396.882757
10         water_and_sanitation          3366           910.723492
11                          NaN          1731          1961.339433
uncategorized projects with null descriptions: 1531


In [ ]:
oecd_updated.loc[(oecd_updated['broad_sector'].isna()) & (oecd_updated['project_description'].isna()), 'broad_sector'] = "Unclassified"
oecd_vis = oecd_updated.dropna(subset=['broad_sector'])
print("this is the amount in USD millions that will not be accounted for in the visualization due to it being uncategorized by the function:",
      oecd_updated.loc[oecd_updated['broad_sector'].isna(), 'usd_disbursements_defl'].sum())

oecd_vis = oecd_vis.drop(columns=['Sector', 'subsector', 'project_description'])
sector_summary(oecd_vis)

this is the amount in USD millions that will not be accounted for in the visualization due to it being uncategorized by the function: 619.5658338089414


,broad_sector,broad_counts,total_disbursements
0,Unclassified,1531,1341.773599
1,aid_and_debt,1642,296.999803
2,economic_development,3817,5763.412615
3,education,16845,7769.019079
4,energy,1403,1434.186233
5,environment_and_agriculture,12246,7977.395595
6,governance_and_social,28104,8337.317484
7,health,36812,27538.565011
8,humanitarian_and_disaster,2453,2159.363519
9,infrastructure,963,606.341395


In [ ]:
oecd_vis.to_csv('oecd_cleaned.csv')